# 03 · Transformer Attention
### Practical LLM Engineering — Module 01: Fundamentals

> **Objectives**
> - The intuition and mathematics behind scaled dot-product attention
> - How multi-head attention works and why multiple heads matter
> - How to implement attention from scratch in PyTorch
> - Masked attention for autoregressive generation
> - Grouped Query Attention (GQA) used in modern LLMs (LLaMA 3, Mistral)
> - Engineering insights: attention complexity, memory costs, and the KV cache preview

---


## 1. Overview

Attention is the engine of the transformer. It answers one question:

> *For each position in the sequence, which other positions are most relevant?*

Before attention, sequence models (RNNs, LSTMs) processed tokens one by one — information had to travel through many time steps to connect distant tokens. Attention solves this by allowing **every token to directly attend to every other token** in a single operation.

```
Input sequence:   "The animal didn't cross the street because it was too tired"
                                                              ↑
                                         What does "it" refer to? → "animal"
                                         Attention learns this mapping.
```

The full transformer block stacks:

```
Input embeddings  (seq_len × d_model)
        ↓
  LayerNorm
        ↓
  Multi-Head Attention  ← this notebook
        ↓
  Residual connection
        ↓
  LayerNorm
        ↓
  Feed-Forward Network
        ↓
  Residual connection
        ↓
Output  (seq_len × d_model)
```

This notebook focuses entirely on the **attention sub-layer**.


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install torch transformers matplotlib numpy -q

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42)
np.random.seed(42)

print("Libraries ready")
print(f"   PyTorch : {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Device  : {device}")


## 3. Background: Scaled Dot-Product Attention

### 3.1 The Core Idea

Each token in the sequence produces three vectors:

- **Query** $\mathbf{Q}$: "What am I looking for?"
- **Key** $\mathbf{K}$: "What do I contain?"
- **Value** $\mathbf{V}$: "What do I return if selected?"

These are computed by projecting the input $\mathbf{X} \in \mathbb{R}^{n \times d_{\text{model}}}$:

$$
\mathbf{Q} = \mathbf{X}\mathbf{W}^Q, \quad
\mathbf{K} = \mathbf{X}\mathbf{W}^K, \quad
\mathbf{V} = \mathbf{X}\mathbf{W}^V
$$

where $\mathbf{W}^Q, \mathbf{W}^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$ and $\mathbf{W}^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$.

---

### 3.2 Scaled Dot-Product Attention

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}
$$

Step by step:

1. **Similarity scores:** $\mathbf{S} = \mathbf{Q}\mathbf{K}^\top \in \mathbb{R}^{n \times n}$ — how much each query attends to each key
2. **Scale:** divide by $\sqrt{d_k}$ to prevent softmax saturation
3. **Softmax:** convert scores to probabilities (rows sum to 1)
4. **Weighted sum:** multiply by $\mathbf{V}$ to get the output

**Why scale by $\sqrt{d_k}$?**

For large $d_k$, the dot products grow in magnitude. If $q_i, k_j \sim \mathcal{N}(0,1)$, then:

$$
\mathbb{E}\left[\mathbf{q} \cdot \mathbf{k}\right] = 0, \quad
\text{Var}\left[\mathbf{q} \cdot \mathbf{k}\right] = d_k
$$

Without scaling, large variance pushes softmax into saturation (near-zero gradients). Dividing by $\sqrt{d_k}$ restores unit variance.


## 4. Scaled Dot-Product Attention — From Scratch

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,   # (batch, heads, seq_q, d_k)
    K: torch.Tensor,   # (batch, heads, seq_k, d_k)
    V: torch.Tensor,   # (batch, heads, seq_k, d_v)
    mask: torch.Tensor = None,  # (batch, 1, seq_q, seq_k) optional
    dropout_p: float = 0.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Scaled dot-product attention.
    Returns: output (same shape as Q with last dim = d_v), attention weights.
    """
    d_k = Q.size(-1)

    # Step 1: Compute raw attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # scores shape: (batch, heads, seq_q, seq_k)

    # Step 2: Apply mask (if provided)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 3: Softmax over key dimension
    attn_weights = F.softmax(scores, dim=-1)

    # Step 4: Dropout (training only)
    if dropout_p > 0.0:
        attn_weights = F.dropout(attn_weights, p=dropout_p)

    # Step 5: Weighted sum of values
    output = torch.matmul(attn_weights, V)
    # output shape: (batch, heads, seq_q, d_v)

    return output, attn_weights


# ── Quick test ─────────────────────────────────────────────────────────
batch, seq_len, d_k, d_v = 1, 6, 64, 64

Q = torch.randn(batch, 1, seq_len, d_k)
K = torch.randn(batch, 1, seq_len, d_k)
V = torch.randn(batch, 1, seq_len, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Input  Q shape : {Q.shape}")
print(f"Output shape   : {output.shape}")
print(f"Attn weights   : {weights.shape}  (sums to 1 along last dim)")
print(f"Row sum check  : {weights[0, 0, 0].sum().item():.6f}  ← should be 1.0")


## 5. Visualising Attention Weights

In [ ]:
def visualize_attention(attn_weights: np.ndarray,
                       tokens: list[str],
                       title: str = "Attention Weights"):
    """Plot an attention heatmap for a single head."""
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(attn_weights, cmap="Blues", vmin=0, vmax=attn_weights.max())

    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=11)
    ax.set_yticklabels(tokens, fontsize=11)
    ax.set_xlabel("Key (attending to)", fontsize=11)
    ax.set_ylabel("Query (from position)", fontsize=11)
    ax.set_title(title, fontsize=13)

    for i in range(len(tokens)):
        for j in range(len(tokens)):
            ax.text(j, i, f"{attn_weights[i, j]:.2f}",
                    ha="center", va="center", fontsize=9,
                    color="white" if attn_weights[i, j] > attn_weights.max() * 0.6 else "black")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


# ── Construct a toy sequence with known structure ──────────────────────
tokens = ["The", "cat", "sat", "on", "the", "mat"]
n = len(tokens)
d_k = 32

# Hand-craft Q and K so "cat" and "mat" get high similarity
torch.manual_seed(7)
X = torch.randn(1, 1, n, d_k)

# Make positions 1 ("cat") and 5 ("mat") similar
X[0, 0, 5] = X[0, 0, 1] + 0.1 * torch.randn(d_k)  # mat ≈ cat

Wq = torch.randn(d_k, d_k) * 0.1
Wk = torch.randn(d_k, d_k) * 0.1
Wv = torch.eye(d_k)

Q_ = X @ Wq
K_ = X @ Wk
V_ = X @ Wv

_, weights_ = scaled_dot_product_attention(Q_, K_, V_)
visualize_attention(weights_[0, 0].detach().numpy(), tokens,
                    "Toy attention: 'mat' attends strongly to 'cat'")


## 6. Causal (Masked) Attention

During **autoregressive generation**, each token may only attend to previous tokens — not future ones. This is enforced with a **causal mask**:

$$
M_{ij} = \begin{cases} 1 & \text{if } i \geq j \\ 0 & \text{if } i < j \end{cases}
$$

Positions where $M_{ij} = 0$ have their scores set to $-\infty$ before softmax, producing zero attention weight.

```
Position:  0   1   2   3   4
           ↓   ↓   ↓   ↓   ↓
    0     [1,  0,  0,  0,  0]    token 0 can only see itself
    1     [1,  1,  0,  0,  0]    token 1 sees tokens 0-1
    2     [1,  1,  1,  0,  0]    token 2 sees tokens 0-2
    3     [1,  1,  1,  1,  0]    ...
    4     [1,  1,  1,  1,  1]    token 4 sees all
```

This is also called **decoder-style** attention. BERT uses full (bidirectional) attention; GPT uses causal attention.


In [ ]:
def causal_mask(seq_len: int) -> torch.Tensor:
    """Lower-triangular mask: 1 where attention is allowed."""
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
    # shape: (1, 1, seq_len, seq_len)


# ── Visualise causal mask ──────────────────────────────────────────────
seq = 8
mask = causal_mask(seq)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(mask[0, 0], cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Causal mask  (1=attend, 0=blocked)", fontsize=11)
axes[0].set_xlabel("Key position")
axes[0].set_ylabel("Query position")
for i in range(seq):
    for j in range(seq):
        axes[0].text(j, i, str(int(mask[0, 0, i, j].item())),
                     ha="center", va="center", fontsize=10,
                     color="white" if mask[0, 0, i, j] else "gray")

# Masked attention weights
Q_c = torch.randn(1, 1, seq, 32)
K_c = torch.randn(1, 1, seq, 32)
V_c = torch.randn(1, 1, seq, 32)
_, w_causal = scaled_dot_product_attention(Q_c, K_c, V_c, mask=causal_mask(seq))

im = axes[1].imshow(w_causal[0, 0].detach().numpy(), cmap="Blues")
axes[1].set_title("Causal attention weights", fontsize=11)
axes[1].set_xlabel("Key position")
axes[1].set_ylabel("Query position")
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print("Upper triangle is zero — future tokens are invisible to current positions.")


## 7. Multi-Head Attention

A single attention head learns one type of relationship. **Multi-head attention** runs $h$ attention heads in parallel, each with its own projections, allowing the model to jointly attend to information from **different representation subspaces**:

$$
\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,\mathbf{W}^O
$$

where each head is:

$$
\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q,\; \mathbf{K}\mathbf{W}_i^K,\; \mathbf{V}\mathbf{W}_i^V)
$$

**Dimension budget:**

$$
d_k = d_v = \frac{d_{\text{model}}}{h}
$$

So total compute stays the same regardless of number of heads.

| Model | $d_{\text{model}}$ | Heads $h$ | $d_k = d_{\text{model}} / h$ |
|---|---|---|---|
| BERT-base | 768 | 12 | 64 |
| GPT-2 | 768 | 12 | 64 |
| LLaMA-7B | 4096 | 32 | 128 |
| LLaMA-70B | 8192 | 64 | 128 |


In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Full multi-head attention as in Vaswani et al. (2017).
    Supports optional causal masking for autoregressive use.
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads

        # Single fused projection for efficiency (common in practice)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(batch, seq, d_model) → (batch, heads, seq, d_k)"""
        B, S, _ = x.shape
        return x.view(B, S, self.n_heads, self.d_k).transpose(1, 2)

    def merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(batch, heads, seq, d_k) → (batch, seq, d_model)"""
        B, H, S, dk = x.shape
        return x.transpose(1, 2).contiguous().view(B, S, H * dk)

    def forward(self, x: torch.Tensor,
                causal: bool = False) -> tuple[torch.Tensor, torch.Tensor]:
        B, S, _ = x.shape

        Q = self.split_heads(self.W_q(x))   # (B, H, S, d_k)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        mask = causal_mask(S).to(x.device) if causal else None
        attn_out, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout_p=self.dropout if self.training else 0.0
        )

        out = self.W_o(self.merge_heads(attn_out))   # (B, S, d_model)
        return out, attn_weights


# ── Verify shapes ──────────────────────────────────────────────────────
d_model, n_heads, seq_len, batch = 128, 8, 10, 2
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)

x = torch.randn(batch, seq_len, d_model)
out, weights = mha(x, causal=True)

print(f"Input  shape : {x.shape}       (batch={batch}, seq={seq_len}, d_model={d_model})")
print(f"Output shape : {out.shape}   ← same as input")
print(f"Attn weights : {weights.shape}  (batch, heads, seq_q, seq_k)")
print(f"d_k per head : {d_model // n_heads}")

total_params = sum(p.numel() for p in mha.parameters())
print(f"\nTotal MHA parameters : {total_params:,}")
print(f"  = 4 × d_model² = 4 × {d_model}² = {4 * d_model**2:,}  ✓")


In [ ]:
# ── Visualise all 8 heads ──────────────────────────────────────────────
sentence = "the cat sat on the mat"
tokens   = sentence.split()
n_tok    = len(tokens)

mha_vis = MultiHeadAttention(d_model=64, n_heads=4)
x_vis   = torch.randn(1, n_tok, 64)
_, head_weights = mha_vis(x_vis, causal=False)   # (1, 4, 6, 6)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Multi-Head Attention: each head learns different patterns", fontsize=12)

for h, ax in enumerate(axes):
    w = head_weights[0, h].detach().numpy()
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=w.max())
    ax.set_title(f"Head {h+1}", fontsize=11)
    ax.set_xticks(range(n_tok)); ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_tok)); ax.set_yticklabels(tokens, fontsize=9)
    if h == 0: ax.set_ylabel("Query")

plt.tight_layout()
plt.show()


## 8. Real Attention Maps from a Pretrained Model

In [ ]:
# ── Extract attention weights from BERT ───────────────────────────────
from transformers import BertTokenizer, BertModel

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model     = BertModel.from_pretrained("bert-base-uncased",
                                            output_attentions=True)
bert_model.eval()

text   = "The animal didn't cross the street because it was too tired."
inputs = bert_tokenizer(text, return_tensors="pt")
tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = bert_model(**inputs)

# outputs.attentions: tuple of (batch, heads, seq, seq) per layer
n_layers = len(outputs.attentions)
n_heads  = outputs.attentions[0].shape[1]
print(f"BERT: {n_layers} layers × {n_heads} heads")
print(f"Sequence tokens: {tokens}")


In [ ]:
# ── Plot layer 11, head 0  (known coreference head in BERT) ──────────
layer_idx = 11
head_idx  = 3

attn = outputs.attentions[layer_idx][0, head_idx].detach().numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attn, cmap="Blues")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90, fontsize=9)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=9)
ax.set_title(f"BERT Layer {layer_idx+1}, Head {head_idx+1}\n"
             f"Notice how 'it' attends strongly to 'animal'", fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# ── Average attention across all heads in last layer ─────────────────
avg_attn = outputs.attentions[-1][0].mean(dim=0).detach().numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(avg_attn, cmap="Purples")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90, fontsize=9)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=9)
ax.set_title("BERT Last Layer — Average across all 12 heads", fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()


## 9. Engineering Insights

### 9.1 Attention Complexity

The attention score matrix $\mathbf{S} = \mathbf{Q}\mathbf{K}^\top$ has shape $(n \times n)$.

| Operation | Time complexity | Space complexity |
|---|---|---|
| Score matrix $\mathbf{QK}^\top$ | $\mathcal{O}(n^2 d_k)$ | $\mathcal{O}(n^2)$ |
| Softmax | $\mathcal{O}(n^2)$ | $\mathcal{O}(n^2)$ |
| Output $\text{softmax} \cdot \mathbf{V}$ | $\mathcal{O}(n^2 d_v)$ | $\mathcal{O}(n^2)$ |

**The $\mathcal{O}(n^2)$ memory bottleneck** is the central engineering challenge for long contexts.

- At $n = 1024$: attention matrix is $1024^2 = 1\text{M}$ elements → fine
- At $n = 8192$: $67\text{M}$ elements per head → getting large
- At $n = 128\text{K}$: $16\text{B}$ elements per head → impossible naively

**FlashAttention** (Dao et al., 2022) solves this by tiling the computation to avoid materialising the full $n \times n$ matrix, achieving $\mathcal{O}(n)$ memory.


In [ ]:
# ── Attention memory cost vs sequence length ─────────────────────────
import matplotlib.ticker as mticker

seq_lengths  = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
n_heads      = 32
bytes_per_el = 2   # float16

memory_GB = []
for n in seq_lengths:
    # n×n attention matrix per head, all heads
    elements  = n * n * n_heads
    mem_bytes = elements * bytes_per_el
    memory_GB.append(mem_bytes / 1024**3)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([str(n) for n in seq_lengths], memory_GB, "o-", color="#e74c3c", linewidth=2, markersize=7)
ax.axhline(y=40,  color="gray", linestyle="--", alpha=0.6, label="A100 40GB VRAM")
ax.axhline(y=80,  color="blue", linestyle="--", alpha=0.6, label="A100 80GB VRAM")
ax.set_xlabel("Sequence length (tokens)", fontsize=11)
ax.set_ylabel("Attention matrix memory (GB, float16)", fontsize=11)
ax.set_title(f"Attention memory grows as O(n²)  —  {n_heads} heads, float16", fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print("\nMemory breakdown:")
print(f"{'Seq len':>10}  {'Memory (GB)':>12}  {'Fits A100-80GB?':>16}")
print("-" * 42)
for n, mem in zip(seq_lengths, memory_GB):
    fits = "✅" if mem < 80 else "❌"
    print(f"{n:>10,}  {mem:>12.3f}  {fits:>16}")


### 9.2 Grouped Query Attention (GQA)

Standard multi-head attention has $h$ query heads AND $h$ key/value heads, giving large KV caches at inference.

**Grouped Query Attention** (Ainslie et al., 2023) used in LLaMA 2/3, Mistral, and Gemma:

- $h$ query heads (same as before)
- $g$ key/value head **groups** where $g < h$
- Each group of $h/g$ query heads shares one K and V head

$$
\text{head}_i = \text{Attention}\!\left(\mathbf{Q}_i\mathbf{W}_i^Q,\; \mathbf{K}_{\lfloor i \cdot g/h \rfloor}\mathbf{W}^K,\; \mathbf{V}_{\lfloor i \cdot g/h \rfloor}\mathbf{W}^V\right)
$$

Special cases:
- $g = h$: standard **Multi-Head Attention (MHA)**
- $g = 1$: **Multi-Query Attention (MQA)** — all queries share one K/V
- $1 < g < h$: **Grouped Query Attention (GQA)**

**KV cache size** at inference:

$$
\text{KV cache} = 2 \times g \times d_k \times n_{\text{layers}} \times n_{\text{tokens}} \times \text{bytes}
$$

GQA reduces KV cache by a factor of $h/g$.


In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention (GQA) as used in LLaMA 2/3, Mistral.
    n_kv_heads < n_heads: each KV head is shared by (n_heads // n_kv_heads) query heads.
    """
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_k        = d_model // n_heads

        self.W_q = nn.Linear(d_model, n_heads    * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model,               bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, D = x.shape

        Q = self.W_q(x).view(B, S, self.n_heads,    self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.n_kv_heads, self.d_k).transpose(1, 2)

        # Expand K and V to match the number of query heads
        # (B, n_kv_heads, S, d_k) → (B, n_heads, S, d_k)
        K = K.repeat_interleave(self.n_groups, dim=1)
        V = V.repeat_interleave(self.n_groups, dim=1)

        out, _ = scaled_dot_product_attention(Q, K, V, mask=causal_mask(S))
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)


# ── Compare MHA vs GQA vs MQA parameter counts & KV cache ─────────────
d_model = 4096   # LLaMA-7B
n_heads = 32

configs = {
    "MHA  (h=32, kv=32)": (32, 32),
    "GQA  (h=32, kv=8) ": (32,  8),
    "MQA  (h=32, kv=1) ": (32,  1),
}

print(f"{'Config':<26} {'KV heads':>9} {'KV params':>12} {'KV cache reduction':>20}")
print("-" * 72)
for name, (nq, nkv) in configs.items():
    d_k        = d_model // nq
    kv_params  = 2 * nkv * d_k * d_model   # W_k + W_v
    reduction  = 32 / nkv
    print(f"{name:<26} {nkv:>9} {kv_params:>12,} {f'{reduction:.0f}×':>20}")


In [ ]:
# ── KV cache size comparison ──────────────────────────────────────────
n_layers   = 32        # LLaMA-7B
d_k        = 128       # d_model / n_heads = 4096 / 32
bytes_fp16 = 2
seq_lengths_kv = [512, 1024, 2048, 4096, 8192]

print("KV Cache size (float16, 32 layers, batch=1)")
print(f"{'Seq len':>10}", end="")
for name, (nq, nkv) in configs.items():
    print(f"  {name.strip():>22}", end="")
print()
print("-" * 85)

for seq in seq_lengths_kv:
    print(f"{seq:>10,}", end="")
    for name, (nq, nkv) in configs.items():
        kv_bytes = 2 * nkv * d_k * n_layers * seq * bytes_fp16
        kv_mb    = kv_bytes / 1024**2
        print(f"  {kv_mb:>21.1f}MB", end="")
    print()


## 10. Putting It Together: A Full Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    """
    Single transformer decoder block:
      LayerNorm → Multi-Head Attention → Residual
      LayerNorm → Feed-Forward Network  → Residual
    """
    def __init__(self, d_model: int, n_heads: int,
                 d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model

        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm attention with residual
        attn_out, _ = self.attn(self.norm1(x), causal=True)
        x = x + attn_out

        # Pre-norm FFN with residual
        x = x + self.ff(self.norm2(x))
        return x


# ── Parameter count breakdown ──────────────────────────────────────────
d_model, n_heads = 512, 8
block = TransformerBlock(d_model=d_model, n_heads=n_heads)

attn_params = sum(p.numel() for p in block.attn.parameters())
ff_params   = sum(p.numel() for p in block.ff.parameters())
norm_params = sum(p.numel() for p in block.norm1.parameters()) * 2
total       = attn_params + ff_params + norm_params

print(f"Transformer Block (d_model={d_model}, n_heads={n_heads})")
print(f"  Attention params    : {attn_params:>10,}  ({attn_params/total*100:.1f}%)")
print(f"  Feed-forward params : {ff_params:>10,}  ({ff_params/total*100:.1f}%)")
print(f"  LayerNorm params    : {norm_params:>10,}  ({norm_params/total*100:.1f}%)")
print(f"  Total               : {total:>10,}")

# Forward pass
x_in  = torch.randn(2, 16, d_model)
x_out = block(x_in)
print(f"\nForward pass: {x_in.shape} → {x_out.shape}  ✅")


## 11. FlashAttention — A Note

Standard attention materialises the full $n \times n$ score matrix in HBM (GPU memory), making memory the bottleneck for long contexts.

**FlashAttention** (Dao et al., 2022) reorders the computation using tiling:

```
Standard attention:
  HBM read  Q, K, V
  Compute   S = QKᵀ        ← write n×n to HBM  ← bottleneck
  Compute   P = softmax(S) ← read/write n×n from HBM
  Compute   O = PV          ← write to HBM

FlashAttention:
  Loop over tiles of Q, K, V in SRAM
  Compute partial softmax with online normalisation
  Never write n×n matrix to HBM
  → O(n) HBM memory, same numerical output
```

FlashAttention is now the default in PyTorch (`F.scaled_dot_product_attention`) and all major LLM frameworks.


In [ ]:
# ── PyTorch native FlashAttention (uses CUDA kernel when available) ───
B, H, S, d_k = 2, 8, 256, 64

Q_fa = torch.randn(B, H, S, d_k, device=device, dtype=torch.float16)
K_fa = torch.randn(B, H, S, d_k, device=device, dtype=torch.float16)
V_fa = torch.randn(B, H, S, d_k, device=device, dtype=torch.float16)

# PyTorch's built-in SDPA (uses FlashAttention on CUDA)
with torch.backends.cuda.sdp_kernel(
    enable_flash=True, enable_math=True, enable_mem_efficient=True
):
    out_flash = F.scaled_dot_product_attention(Q_fa, K_fa, V_fa, is_causal=True)

print(f"FlashAttention output shape : {out_flash.shape}")
print(f"Device used                 : {out_flash.device}")
print()
print("PyTorch selects the best attention kernel automatically:")
print("  - CUDA GPU  → FlashAttention (fast, O(n) memory)")
print("  - CPU       → Math attention (reference implementation)")


## 12. Exercises

1. **Scaling experiment:** Implement the attention score matrix for sequence lengths `[128, 512, 1024, 4096]`. Time each with `torch.cuda.Event`. Plot time and memory vs sequence length. Does it match the $\mathcal{O}(n^2)$ prediction?

2. **Head specialisation:** Load BERT and visualise all 12 heads in layer 6 for the sentence *"The bank can guarantee deposits will cover future tuition costs."* Do any heads distinguish the two senses of "bank"?

3. **GQA correctness:** Verify that your `GroupedQueryAttention` with `n_kv_heads = n_heads` produces the same output as `MultiHeadAttention` (up to floating point tolerance) when weights are identical.

4. **Attention entropy:** Compute the entropy $H = -\sum_j w_j \log w_j$ of each attention row. High entropy = diffuse attention; low entropy = sharp focus. Which layers/heads in BERT have the highest entropy? What do these "broad attention" heads do?

5. **Causal vs bidirectional:** Encode the same sentence with causal masking and without. Compute the cosine similarity between the output hidden states for each token. How much does future context change a token's representation?

---

## 13. Key Takeaways

| Concept | Summary |
|---|---|
| **Q, K, V** | Three projections per token: what I want, what I have, what I give |
| **Scaled dot-product** | $\text{softmax}(\mathbf{QK}^\top / \sqrt{d_k})\mathbf{V}$ |
| **Scaling by $\sqrt{d_k}$** | Prevents softmax saturation for large $d_k$ |
| **Causal mask** | Blocks future tokens; required for autoregressive generation |
| **Multi-head** | Parallel heads capture different relationship types |
| **GQA** | Fewer KV heads → smaller KV cache at inference, used in LLaMA 3 / Mistral |
| **$\mathcal{O}(n^2)$ cost** | Core bottleneck for long contexts; solved by FlashAttention |
| **FlashAttention** | Tiled computation → $\mathcal{O}(n)$ memory, same output |
